# 03 — Phan tach Shapley-Shorrocks

Notebook nay chay phan phan tach cua SIFT tren lattice 16 cau hinh.

**Ham gia tri theo dang khoang cach** (thiet ke muc 8, ban R3):

$$\Delta(S) = M_{\text{reference}}(S) - M_{\text{temporal}}(S), \qquad
v(S) = \Delta(\varnothing) - \Delta(S), \qquad R = \Delta(N)$$

Nhanh tham chieu la `random_fully_matched` va no **di chuyen theo coalition**:
`b2_axis` doi tap ho ma cua so temporal chua, nen tham chieu phai gioi han tap
kiem tra vao dung tap ho do. Day chinh la cach xu ly phan bien ve tinh so sanh
duoc cua $v(S)$ neu o R0.

Chi hai thiet ke `temporal` va `random_fully_matched` mang du 16 coalition. Hai
thiet ke `random` va `random_matched` chi chay o cau hinh nen va **khong bao gio**
duoc dua vao `exact_shapley`; chung phuc vu phep tach ba phan o muc 7bis.

Dong nhat thuc loi, dung theo cau tao chu khong phai mot phat hien thuc nghiem:

$$\Delta(\varnothing) = \phi_{A1} + \phi_{A2} + \phi_{B1} + \phi_{B2} + R$$

**Day khong phai SHAP.** Trich dan phuong phap la Shapley (1953) va Shorrocks (2013).

---

*Trang thai:* notebook da chay tron ven tren mot bang tong hop dung schema san
xuat (3.400 fit, 780.000 dong prediction) de kiem duong ong. **Chua chay tren so
lieu that**; moi o deu trong output. Khi `results/metrics.parquet` va
`results/predictions.parquet` cua E2 co mat thi chay lai tu dau, khong sua gi.*

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import warnings

import numpy as np
import pandas as pd

from sift import cache
from sift.config import CONTROL_NAMES, PanelSpec
from sift.data import build_panel, load_mlran
from sift.experiment import lattice_cells, reference_cells
from sift.paths import RESULTS_DIR
from sift.reporting import (
    CONTROL_LABELS,
    build_agreement_report,
    build_coalition_values,
    build_gap_decomposition,
    build_gap_table,
    build_interaction_table,
    build_lattice_table,
    build_model_summary,
    MIN_COMMON_SAMPLES,
    build_verification_table,
    compare_metric_decompositions,
    class_prior_reference,
    panel_matches_run,
    decompose_groups,
    rescore_predictions,
    verify_a1_recomputation,
    to_booktabs,
)
from sift.shapley import (
    check_efficiency,
    check_mobius_consistency,
    check_owen_group_consistency,
    check_weights_sum_to_one,
    exact_shapley,
    harsanyi_dividends,
    owen_value,
)

# sklearn warns once per call when a family is predicted but absent from y_true;
# that is the expected shape of a 32-class temporal window, not a fault.
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

TIERS = [("a1_prior", "a2_labels"), ("b1_fs", "b2_axis")]
TIER_NAMES = ("data_side", "protocol_side")
PHI_COLUMNS = [f"phi_{name}" for name in CONTROL_NAMES]
PRIMARY_METRIC = "macro_f1"
# Section 3/A1: macro-F1 averages over classes and can neutralise a shift in
# the class prior, which is the one thing A1 manipulates. accuracy is
# prior-sensitive. Decomposing under both stops the metric deciding which
# confounds appear to exist.
METRICS_TO_DECOMPOSE = ("macro_f1", "accuracy")

## 1. Nap du lieu va tach vai tro

`lattice_cells` va `reference_cells` loc theo cot `role` do E2 ghi. Khong loc tay
theo danh sach ten thiet ke: neu them mot thiet ke moi thi cot `role` la cho duy
nhat phai sua.

In [3]:
metrics = pd.read_parquet(RESULTS_DIR / "metrics.parquet")
predictions = pd.read_parquet(RESULTS_DIR / "predictions.parquet")

lattice = lattice_cells(metrics)
reference = reference_cells(metrics)

print(f"metrics      {metrics.shape[0]:>7,} rows")
print(f"predictions  {predictions.shape[0]:>7,} rows")
print(f"lattice      {lattice.shape[0]:>7,} rows  designs={sorted(lattice['design'].unique())}")
print(f"reference    {reference.shape[0]:>7,} rows  designs={sorted(reference['design'].unique())}")
display(metrics.groupby(["design", "role"]).size().rename("n_fits").to_frame())

metrics        3,400 rows
predictions  722,880 rows
lattice        3,200 rows  designs=['random_fully_matched', 'temporal']
reference        200 rows  designs=['random', 'random_matched']


,,n_fits
design,role,
random,reference,100
random_fully_matched,lattice,1600
random_matched,reference,100
temporal,lattice,1600


## 1b. Phep tach ba phan cua khoang cach da bao cao (muc 7bis)

Truoc khi lattice chay, khoang cach ma cac paper van bao cao duoc tach lam ba:

```
size effect              = Delta(random) - Delta(random_matched)
class-composition effect = Delta(random_matched) - Delta(random_fully_matched)
remainder                = Delta(random_fully_matched)
```

Ba so hang cong dung bang `Delta(random)`, tuc dong nhat thuc chu khong phai ket
qua do. Phan `remainder` moi la thu lattice phan tach.

**Ca hai so hang hieu chinh deu co dau.** Thanh phan lop do duoc -0,033 tai cut
2023 va +0,237 tai cut 2015; so hang co mau am o moi cut ma temporal huan luyen
tren nhieu du lieu hon tham chieu naive. Khong duoc cat ve khong o bat ky dau.

In [4]:
three_way = []
for (cut_value, model_name), _ in metrics.groupby(["cut", "model"], sort=True):
    terms = build_gap_decomposition(metrics, model=model_name, cut=int(cut_value))
    three_way.append({"cut": cut_value, "model": model_name, **terms})
three_way = pd.DataFrame(three_way)
three_way["telescoping_error"] = (
    three_way[["size", "class_composition", "remainder"]].sum(axis=1)
    - three_way["delta_reported"]
)
display(three_way.style.format({c: "{:+.4f}" for c in three_way.columns[2:]}))

print(f"max |telescoping error| = {three_way['telescoping_error'].abs().max():.3e}")
print(f"size term range          {three_way['size'].min():+.4f} .. {three_way['size'].max():+.4f}")
print(f"class term range         {three_way['class_composition'].min():+.4f} .. "
      f"{three_way['class_composition'].max():+.4f}")
print(f"remainder range          {three_way['remainder'].min():+.4f} .. "
      f"{three_way['remainder'].max():+.4f}")
for name in ("size", "class_composition"):
    if (three_way[name] < 0).any():
        print(f"note: {name} is negative in "
              f"{int((three_way[name] < 0).sum())} of {len(three_way)} cells, "
              "which is expected and must not be clipped")

,cut,model,size,class_composition,remainder,delta_reported,telescoping_error
0,2015,lightgbm,+0.0409,+0.0745,+0.4385,+0.5540,+0.0000
1,2015,logreg,+0.0472,+0.0757,+0.4737,+0.5967,+0.0000
2,2015,mlp,+0.0477,+0.0814,+0.4480,+0.5771,+0.0000
3,2015,random_forest,+0.0451,+0.0937,+0.4481,+0.5868,+0.0000
4,2017,lightgbm,+0.0214,-0.0175,+0.5291,+0.5330,+0.0000
5,2017,logreg,+0.0144,-0.0272,+0.5752,+0.5624,+0.0000
6,2017,mlp,+0.0026,-0.0064,+0.5131,+0.5093,+0.0000
7,2017,random_forest,+0.0270,-0.0180,+0.5443,+0.5533,+0.0000
8,2019,lightgbm,-0.0085,-0.0073,+0.5423,+0.5264,+0.0000
9,2019,logreg,-0.0152,-0.0065,+0.5967,+0.5750,+0.0000


max |telescoping error| = 0.000e+00
size term range          -0.0152 .. +0.1133
class term range         -0.1266 .. +0.0937
remainder range          +0.2300 .. +0.5967
note: size is negative in 5 of 20 cells, which is expected and must not be clipped
note: class_composition is negative in 12 of 20 cells, which is expected and must not be clipped


## 1c. Trong so cua control A1: tinh lai, khong doc tu dia

Control A1 ap trong so vao **luc tinh metric** va vector trong so do khong duoc
luu trong `predictions.parquet`. No khong can duoc luu: trong so cua mot mau la

$$w_c = rac{p_{	ext{reference}}(c)}{p_{	ext{window}}(c)}$$

chuan hoa lai de trung binh bang mot. Phan tham chieu la phan phoi lop cua **toan
panel**, khong phu thuoc coalition, cut hay seed; phan window suy ra tu chinh
`y_true`, von co san trong bang prediction. Vi vay trong so tai lap duoc day du.

**Tinh lai la tien loi; tai lap duoc con so da luu moi la dang tin.** O duoi kiem
dung dieu do: lay mot mau cac o co A1 bat, tinh lai trong so, cham lai metric, roi
doi chieu voi `macro_f1` da luu. Neu co dong nao lech thi trong so khong phai thu
da sinh ra so da cong bo, va **khong duoc bao cao attribution cua A1** cho toi khi
viec do duoc giai quyet.

In [5]:
panel, provenance = build_panel(load_mlran(), PanelSpec())
prior_reference = class_prior_reference(panel)
print(f"panel {panel.shape}  families {panel['ransomware_family'].nunique()}")
print(f"reference prior over {len(prior_reference)} classes, "
      f"sums to {prior_reference.sum():.12f}")

# Everything recomputed below depends on this being the panel the run used.
manifest_path = RESULTS_DIR / "run_manifest.json"
manifest = (json.loads(manifest_path.read_text(encoding="utf-8"))
            if manifest_path.exists() else {})
if not manifest:
    print("!!! no run_manifest.json; the panel cannot be checked against the run")
provenance_check = panel_matches_run(panel, manifest)
print(provenance_check["detail"])
if not provenance_check["matches"]:
    raise AssertionError(
        "the panel rebuilt here is not the one results/ was produced from; "
        "recomputed A1 weights and common-basis windows would not correspond "
        "to the stored metrics. Rebuild the panel with the run's PanelSpec, or "
        "re-read the results directory."
    )

panel (1377, 504)  families 32
reference prior over 32 classes, sums to 1.000000000000
panel matches the recorded run (0e5c02de829e5b15)


In [6]:
a1_check = verify_a1_recomputation(predictions, metrics, prior_reference,
                                   sample_size=40)
display(a1_check.head(10))

n_disagree = int((~a1_check["agrees"]).sum())
print(f"checked {len(a1_check)} A1-on fits, max |diff| = {a1_check['abs_diff'].max():.3e}")
if n_disagree:
    raise AssertionError(
        f"{n_disagree} A1-on fit(s) could not be reproduced from recomputed weights. "
        "The A1 attribution must not be reported until this is resolved. First "
        "suspect the panel: the reference prior is its class distribution, so a "
        "panel that differs from the run's gives every A1 cell a wrong weight."
    )
print("OK: recomputed A1 weights reproduce the stored metric exactly.")

,fit_id,stored,recomputed,abs_diff,n_test,agrees
0,1e86bdf411a065758c3856f7edfa2229,0.500000,0.500000,0.0,27,True
1,65b4e3f35e8f61607f5758a6aa6912cb,0.779295,0.779295,0.0,27,True
2,159d935203f20166ae70e0bfc52b8b8e,0.639770,0.639770,0.0,60,True
3,c60298541669fcb1a8d4665ed4b9f0ba,0.332646,0.332646,0.0,130,True
4,d4a73724b036dcc59be601c279db0fa7,0.425077,0.425077,0.0,324,True
5,87538a3fd50a250fe9fdb2c6dc5d71dd,0.724769,0.724769,0.0,608,True
6,4a9231392fe88a1f9e0b9ff0fc9e4352,0.152380,0.152380,0.0,625,True
7,f8c8162236dda0934b4b0aadf510a7b2,0.144227,0.144227,0.0,625,True
8,ef5e531f87dc54ff786211f210aa9916,0.741166,0.741166,0.0,178,True
9,b5f8563a66a9b6940889d0839f729626,0.686998,0.686998,0.0,64,True


checked 40 A1-on fits, max |diff| = 0.000e+00
OK: recomputed A1 weights reproduce the stored metric exactly.


## 2. Tinh lai moi metric tu bang prediction

Cot metric trong `metrics.parquet` ton tai de **bat lech giua hai file**, khong
phai de lam nguon chan ly. O day tinh lai tung metric tu `predictions.parquet`
roi doi chieu.

Trong so cua A1 duoc **tinh lai** o muc 1c va truyen vao qua `prior_reference`,
nen ca 16 o deu tai lap duoc va `reconstructable` phai bang `True` o moi dong.
Bat ky lech nao bay gio cung la loi that, khong con truong hop duoc mien.

In [7]:
as_reported = rescore_predictions(predictions, metrics, basis="as_reported",
                                  prior_reference=prior_reference)
agreement = build_agreement_report(as_reported, metrics)
display(agreement)

reconstructable = agreement[agreement["reconstructable"]]
n_bad = int(reconstructable["n_mismatch"].sum())
if n_bad:
    print(f"!!! {n_bad} fit(s) disagree between the two files on reconstructable cells")
    print("    Downstream numbers cannot be trusted until this is resolved.")
else:
    print(f"OK: {int(reconstructable['n_fits'].iloc[0]):,} reconstructable fits agree, "
          f"max |diff| = {reconstructable['max_abs_diff'].max():.3e}")

unreconstructable = agreement[~agreement["reconstructable"]]
if not unreconstructable.empty:
    print("!!! some cells are still flagged unreconstructable; prior_reference was")
    print("    not supplied to rescore_predictions and A1 was scored unweighted.")
else:
    print("Every cell, A1-on included, was reproduced from the predictions table.")

,metric,reconstructable,n_fits,n_mismatch,max_abs_diff,worst_fit_id
0,accuracy,True,3400,0,0.0,bc45bce7d64b3c9dfcdbdb735729c551
1,balanced_accuracy,True,3400,0,0.0,bc45bce7d64b3c9dfcdbdb735729c551
2,macro_f1,True,3400,0,0.0,bc45bce7d64b3c9dfcdbdb735729c551
3,mcc,True,3400,0,0.0,bc45bce7d64b3c9dfcdbdb735729c551


OK: 3,400 reconstructable fits agree, max |diff| = 0.000e+00
Every cell, A1-on included, was reproduced from the predictions table.


## 3. Hai co so cham diem (thiet ke muc 8.5)

| Co so | Cach cham | Vai tro |
|---|---|---|
| `as_reported` | moi cau hinh cham tren tap kiem tra cua chinh no | chinh |
| `common` | ca 16 o cham tren giao cua moi tap kiem tra trong nhom | phep kiem tinh so sanh duoc |

Tren co so `common`, `y_true` giong het nhau o ca 16 o nen tap lop tu dong bi
dong bang, khong can ghim them.

Tren co so `common`, trong so A1 duoc tinh lai tu nhan cua **cua so giao** chu
khong phai cua so day du, vi ty le lop cua mot tap con khac di va trong so la ham
cua chinh cac ty le do. Nghia la A1 van co hieu luc day du tren ca hai co so.

In [8]:
common = rescore_predictions(predictions, metrics, basis="common",
                             prior_reference=prior_reference)

feasibility = (
    common.merge(metrics[["fit_id", "role"]], on="fit_id")
    .groupby(["role", "design", "cut"])
    .agg(n_cells=("fit_id", "size"),
         n_common_min=("n_common", "min"),
         n_common_max=("n_common", "max"),
         n_infeasible=("common_feasible", lambda s: int((~s).sum())))
)
display(feasibility)

n_infeasible = int(feasibility["n_infeasible"].sum())
if n_infeasible:
    print(f"{n_infeasible} cells share fewer than {MIN_COMMON_SAMPLES} test samples "
          "across their 16 coalitions.")
    print("The matched random arm redraws its test set at every coalition, so its")
    print("windows overlap only by chance; the temporal arm's window is determined")
    print("by the time axis and overlaps far more. Groups containing such a cell")
    print("are excluded from the common-basis decomposition and counted below.")
else:
    print("OK: every group has a usable common test window.")

n_cells  n_common_min  n_common_max  n_infeasible
role      design               cut                                                    
lattice   random_fully_matched 2015      320             8            20           256
                               2017      320             8            14           320
                               2019      320            36            46             0
                               2021      320            23            42             0
                               2023      320            15            17           320
          temporal             2015      320            66            66             0
                               2017      320            50            50             0
                               2019      320            61            61             0
                               2021      320            94            94             0
                               2023      320            27            27             0
reference random               2015       20           345           345             0
                               2017       20           345           345             0
                               2019       20           345           345             0
                               2021       20           345           345             0
                               2023       20           345           345             0
          random_matched       2015       20           290           290             0
                               2017       20           232           232             0
                               2019       20           268           268             0
                               2021       20           130           130             0
                               2023       20            27            27             0

896 cells share fewer than 20 test samples across their 16 coalitions.
The matched random arm redraws its test set at every coalition, so its
windows overlap only by chance; the temporal arm's window is determined
by the time axis and overlaps far more. Groups containing such a cell
are excluded from the common-basis decomposition and counted below.


## 4. Xay ham gia tri $v(S)$ tren ca hai co so

`build_gap_table` ghep hai nhanh theo `(cut, model, seed, config_id)` va nem loi
neu mot nhom nao do thieu o. Khong co o nao duoc bo qua am tham.

In [9]:
roles = metrics[["fit_id", "role"]]
decompositions, frames, values_by_key, skipped_by_key = {}, {}, {}, {}

for basis, scored in (("as_reported", as_reported), ("common", common)):
    cells = scored.merge(roles, on="fit_id", validate="one_to_one")
    cells = cells[cells["role"] == "lattice"]
    # as-reported must be complete; the common basis may lose groups whose two
    # arms share too few samples, and those are reported rather than dropped.
    policy = "raise" if basis == "as_reported" else "skip"
    for metric_name in METRICS_TO_DECOMPOSE:
        gaps = build_gap_table(cells, metric_column=metric_name)
        vals = build_coalition_values(gaps)
        decs_k, frame_k, skipped_k = decompose_groups(
            vals, TIERS, TIER_NAMES, on_incomplete=policy)
        key = (basis, metric_name)
        values_by_key[key], decompositions[key] = vals, decs_k
        frames[key], skipped_by_key[key] = frame_k, skipped_k
        print(f"{basis:>12} / {metric_name:<10}: {len(decs_k)} instances decomposed, "
              f"{len(skipped_k)} skipped, "
              f"checks passed = {bool(frame_k['checks_passed'].all())}")

# The primary view; every later section that does not say otherwise uses it.
skipped_common = skipped_by_key[("common", PRIMARY_METRIC)]
if len(skipped_common):
    print()
    print(f"common basis lost {len(skipped_common)} of 100 groups:")
    display(skipped_common.groupby("cut").size().rename("n_groups").to_frame())

values = values_by_key[("as_reported", PRIMARY_METRIC)]
frame = frames[("as_reported", PRIMARY_METRIC)]
decs = decompositions[("as_reported", PRIMARY_METRIC)]

 as_reported / macro_f1  : 100 instances decomposed, 0 skipped, checks passed = True
 as_reported / accuracy  : 100 instances decomposed, 0 skipped, checks passed = True


      common / macro_f1  : 44 instances decomposed, 56 skipped, checks passed = True
      common / accuracy  : 44 instances decomposed, 56 skipped, checks passed = True

common basis lost 56 of 100 groups:


,n_groups
cut,
2015,16
2017,20
2023,20


## 5. Bon phep kiem so hoc, in ra chu khong giau trong assert

Ba phep kiem bat buoc o muc 8.3 cong them phep kiem nhat quan nhom cua Owen.
Kiem Moebius la duong tinh hoan toan khac nen no bat duoc loi ma efficiency bo sot.

In [10]:
sample_key = sorted(decs)[0]
sample_rows = values[(values["cut"] == sample_key[0])
                     & (values["model"] == sample_key[1])
                     & (values["seed"] == sample_key[2])]
sample_values = dict(zip(sample_rows["coalition"], sample_rows["v"]))
phi = exact_shapley(sample_values)
owen = owen_value(sample_values, TIERS)
dividends = harsanyi_dividends(sample_values)

print(f"instance (cut, model, seed) = {sample_key}")
display(build_verification_table([
    check_weights_sum_to_one(len(CONTROL_NAMES)),
    check_efficiency(sample_values, phi),
    check_mobius_consistency(sample_values, phi),
    check_owen_group_consistency(sample_values, TIERS, owen),
]))

instance (cut, model, seed) = (np.int16(2015), 'lightgbm', np.int16(0))


,check,passed,deviation,tolerance,detail
0,weights_sum_to_one,True,0.000000e+00,1.000000e-09,weighted subset counts sum to 1.0 for n_players=4
1,efficiency,True,0.000000e+00,1.000000e-09,sum(phi) = 0.20448880019027704 against v(N) = ...
2,mobius_consistency,True,3.469447e-18,1.000000e-09,largest discrepancy at 'a1_prior': direct 0.01...
3,owen_group_consistency,True,2.775558e-17,1.000000e-09,largest union discrepancy at 'tier_0': 2.776e-17


In [11]:
# The same four checks over every instance, on both bases.
rows = []
for (basis, metric_name), decs_all in decompositions.items():
    for key, result in decs_all.items():
        for check in result.checks:
            rows.append({"basis": basis, "metric": metric_name, "check": check.name,
                         "passed": check.passed, "deviation": check.deviation})
check_summary = (
    pd.DataFrame(rows)
    .groupby(["basis", "metric", "check"])
    .agg(n=("passed", "size"),
         n_failed=("passed", lambda s: int((~s).sum())),
         max_deviation=("deviation", "max"))
)
display(check_summary)
print("all checks passed:", bool(check_summary["n_failed"].sum() == 0))

n  n_failed  max_deviation
basis       metric   check                                               
as_reported accuracy efficiency              100         0   5.551115e-17
                     efficiency_owen         100         0   2.775558e-17
                     mobius_consistency      100         0   5.551115e-17
                     owen_group_consistency  100         0   5.551115e-17
                     weights_sum_to_one      100         0   0.000000e+00
            macro_f1 efficiency              100         0   5.551115e-17
                     efficiency_owen         100         0   5.551115e-17
                     mobius_consistency      100         0   2.775558e-17
                     owen_group_consistency  100         0   2.775558e-17
                     weights_sum_to_one      100         0   0.000000e+00
common      accuracy efficiency               44         0   5.551115e-17
                     efficiency_owen          44         0   1.387779e-17
                     mobius_consistency       44         0   5.551115e-17
                     owen_group_consistency   44         0   5.551115e-17
                     weights_sum_to_one       44         0   0.000000e+00
            macro_f1 efficiency               44         0   2.775558e-17
                     efficiency_owen          44         0   1.387779e-17
                     mobius_consistency       44         0   2.775558e-17
                     owen_group_consistency   44         0   1.040834e-17
                     weights_sum_to_one       44         0   0.000000e+00

all checks passed: True


In [12]:
# The core identity, verified on every instance rather than asserted once.
for key, current in frames.items():
    recomposed = current[PHI_COLUMNS].sum(axis=1) + current["residual"]
    worst = float(np.max(np.abs(recomposed - current["delta_empty"])))
    # Unscored rows on the common basis are NaN and are excluded above; judging
    # v(empty) on them would report a missing measurement as a broken identity.
    zero_v = values_by_key[key]
    baseline_v = zero_v.loc[zero_v["config_id"] == 0, "v"].dropna()
    zero_ok = bool((baseline_v.abs() < 1e-15).all())
    label = f"{key[0]}/{key[1]}"
    print(f"{label:>26}: max |sum(phi) + R - Delta(empty)| = {worst:.3e}   "
          f"v(empty) == 0 exactly: {zero_ok}")

      as_reported/macro_f1: max |sum(phi) + R - Delta(empty)| = 1.110e-16   v(empty) == 0 exactly: True


      as_reported/accuracy: max |sum(phi) + R - Delta(empty)| = 1.110e-16   v(empty) == 0 exactly: True
           common/macro_f1: max |sum(phi) + R - Delta(empty)| = 1.388e-17   v(empty) == 0 exactly: True
           common/accuracy: max |sum(phi) + R - Delta(empty)| = 5.551e-17   v(empty) == 0 exactly: True


## 6. Bang phan tach theo tung mo hinh

Chi lay trung binh tren seed. **Khong** gop 20 o cut nhan mo hinh thanh mot so:
neu $\phi_{B1}$ troi o LightGBM con $\phi_{A1}$ troi o Logistic Regression thi
so trung binh gay hieu nham, va chinh khac biet do la mot phat hien (muc 8.7).

Khoang o day la do trai theo 5 seed mo hinh. Hai trong bon mo hinh tat dinh nen
do trai cua chung bang dung khong; khoang tin cay cuoi cung phai lay tu bootstrap
tren bang prediction, viec do thuoc notebook 04.

In [13]:
summary = build_model_summary(frame)
columns = (["cut", "model", "n_seeds", "delta_empty"] + PHI_COLUMNS
           + ["residual", "residual_share", "checks_passed"])
display(summary[columns].style.format({c: "{:.4f}" for c in columns[3:-1]}))

,cut,model,n_seeds,delta_empty,phi_a1_prior,phi_a2_labels,phi_b1_fs,phi_b2_axis,residual,residual_share,checks_passed
0,2015,lightgbm,5,0.4385,0.0161,0.1539,-0.0276,0.0789,0.2171,0.4966,True
1,2015,logreg,5,0.4737,0.0269,0.1436,0.0101,0.0465,0.2466,0.5196,True
2,2015,mlp,5,0.4480,0.0278,0.1562,-0.0085,0.0519,0.2208,0.4958,True
3,2015,random_forest,5,0.4481,0.0406,0.1626,-0.0058,0.0564,0.1942,0.4378,True
4,2017,lightgbm,5,0.5291,0.0294,0.2315,-0.0026,0.1441,0.1267,0.2383,True
5,2017,logreg,5,0.5752,0.0494,0.1686,-0.0156,0.0792,0.2937,0.5102,True
6,2017,mlp,5,0.5131,0.0479,0.1958,-0.0733,0.1230,0.2197,0.4359,True
7,2017,random_forest,5,0.5443,0.0532,0.1808,-0.0045,0.1083,0.2066,0.3790,True
8,2019,lightgbm,5,0.5423,0.0281,0.1928,-0.0667,0.1302,0.2577,0.4763,True
9,2019,logreg,5,0.5967,-0.0047,0.1729,-0.0141,0.1390,0.3036,0.5094,True


In [14]:
# Sign of each phi per model across cuts. A control whose sign flips between
# models is a finding, not noise to be averaged away.
for control in CONTROL_NAMES:
    block = summary.pivot(index="model", columns="cut", values=f"phi_{control}")
    positive = (block > 0).sum(axis=1)
    print(f"phi_{control:<10} positive at {positive.to_dict()} of "
          f"{block.shape[1]} cuts, per model")

phi_a1_prior   positive at {'lightgbm': 5, 'logreg': 3, 'mlp': 5, 'random_forest': 5} of 5 cuts, per model
phi_a2_labels  positive at {'lightgbm': 4, 'logreg': 4, 'mlp': 4, 'random_forest': 4} of 5 cuts, per model
phi_b1_fs      positive at {'lightgbm': 0, 'logreg': 1, 'mlp': 1, 'random_forest': 1} of 5 cuts, per model
phi_b2_axis    positive at {'lightgbm': 5, 'logreg': 5, 'mlp': 4, 'random_forest': 5} of 5 cuts, per model


## 7. Dong gop theo tang (Owen value)

Hai tang la {A1, A2} phia du lieu va {B1, B2} phia quy trinh. Neu phan lon muc
suy giam thuoc tang quy trinh thi cac con so da cong bo trong tai lieu can duoc
danh gia lai, do la ham y chinh cua bai.

In [15]:
tier_columns = [c for c in summary.columns if c.startswith("tier_") and not c.endswith("_half")]
owen_columns = [f"owen_{name}" for name in CONTROL_NAMES if f"owen_{name}" in summary]
display(summary[["cut", "model"] + tier_columns + owen_columns].style.format(
    {c: "{:.4f}" for c in tier_columns + owen_columns}))

share = summary[tier_columns].div(summary[tier_columns].sum(axis=1), axis=0)
print("protocol-side share of v(N), per row (percent):")
print((share["tier_protocol_side"] * 100).round(1).to_list())

,cut,model,tier_data_side,tier_protocol_side,owen_a1_prior,owen_a2_labels,owen_b1_fs,owen_b2_axis
0,2015,lightgbm,0.1675,0.0539,0.0112,0.1564,-0.0296,0.0835
1,2015,logreg,0.1637,0.0634,0.0228,0.1410,0.0121,0.0513
2,2015,mlp,0.1768,0.0505,0.0264,0.1505,-0.0074,0.0579
3,2015,random_forest,0.2025,0.0513,0.0380,0.1645,-0.0074,0.0587
4,2017,lightgbm,0.2408,0.1616,0.0273,0.2135,0.0039,0.1578
5,2017,logreg,0.2190,0.0626,0.0560,0.1630,-0.0155,0.0781
6,2017,mlp,0.2311,0.0624,0.0456,0.1854,-0.0684,0.1308
7,2017,random_forest,0.2378,0.0999,0.0476,0.1903,-0.0110,0.1109
8,2019,lightgbm,0.2374,0.0471,0.0395,0.1980,-0.0747,0.1218
9,2019,logreg,0.1697,0.1234,-0.0037,0.1734,-0.0134,0.1368


protocol-side share of v(N), per row (percent):
[24.3, 27.9, 22.2, 20.2, 40.2, 22.2, 21.3, 29.6, 16.6, 42.1, 44.5, 30.0, 2.3, 32.4, 12.2, 4.3, 73.8, 100.9, 111.5, 96.5]


## 8. Bang phu luc 16 o

Day la bang chung goc, khong phu thuoc bat ky gia dinh tong hop nao. In du 16
dong cho mot mo hinh tai mot cut.

In [16]:
cuts_available = sorted(values["cut"].unique())
LATTICE_CUT = int(cuts_available[len(cuts_available) // 2])
LATTICE_MODEL = sorted(values["model"].unique())[0]
lattice_table = build_lattice_table(values, model=LATTICE_MODEL, cut=LATTICE_CUT)
numeric = ["m_temporal", "m_reference", "delta", "v", "v_std", "share_of_delta"]
display(lattice_table[["config_id", "flags", "active"] + numeric].style.format(
    {c: "{:.4f}" for c in numeric}))

,config_id,flags,active,m_temporal,m_reference,delta,v,v_std,share_of_delta
0,0,0000,none,0.2031,0.7454,0.5423,0.0000,0.0000,0.0000
1,1,1000,A1,0.2490,0.7451,0.4961,0.0462,0.0002,0.0853
2,2,0100,A2,0.2797,0.7541,0.4745,0.0678,0.0445,0.1233
3,3,1100,"A1, A2",0.3323,0.7541,0.4218,0.1205,0.0444,0.2205
4,4,0010,B1,0.1308,0.7446,0.6138,-0.0716,0.0240,-0.1327
5,5,1010,"A1, B1",0.1387,0.7446,0.6059,-0.0636,0.0237,-0.1181
6,6,0110,"A2, B1",0.1812,0.7631,0.5819,-0.0396,0.0393,-0.0746
7,7,1110,"A1, A2, B1",0.1811,0.7631,0.5819,-0.0397,0.0392,-0.0747
8,8,0001,B2,0.1466,0.7135,0.5669,-0.0246,0.0254,-0.0460
9,9,1001,"A1, B2",0.1524,0.7135,0.5611,-0.0189,0.0255,-0.0353


## 9. Bang tuong tac tu Harsanyi dividend

Dividend doi $m(\{i,j\})$ lon so voi cac main effect la bang chung dinh luong
cho viec tat control lan luot phu thuoc thu tu, va tra loi truoc cau hoi cua
reviewer ve viec cac control co tuong tac hay khong.

In [17]:
interaction = build_interaction_table(decs[sample_key].dividends, controls=CONTROL_NAMES)
display(interaction.style.format({"dividend": "{:+.4f}"}))

main_effect = interaction[interaction["order"] == 1]["dividend"].abs().max()
pairwise = interaction[interaction["order"] == 2]["dividend"].abs().max()
print(f"largest |main effect| = {main_effect:.4f}, "
      f"largest |pairwise interaction| = {pairwise:.4f} "
      f"({pairwise / main_effect:.0%} of the main effect)")

,order,coalition,dividend
0,1,A2,+0.1910
1,1,B2,+0.0445
2,1,B1,-0.0387
3,1,A1,+0.0094
4,2,A2 + B1,-0.0766
5,2,A2 + B2,-0.0496
6,2,B1 + B2,+0.0473
7,2,A1 + A2,-0.0254
8,2,A1 + B1,-0.0077
9,2,A1 + B2,+0.0026


largest |main effect| = 0.1910, largest |pairwise interaction| = 0.0766 (40% of the main effect)


## 9b. Lua chon metric co the che mot confound (muc 3/A1)

Tai lieu thiet ke lap luan rang **macro-F1 co the trung hoa $\phi_{A1}$ ve gan
khong**. Ly do la macro-F1 lay trung binh deu tren cac lop nen gan nhu mien nhiem
voi thay doi ty le lop, ma ty le lop lai la dung thu A1 tac dong. accuracy thi
nhay voi prior nen lo ra tac dong do. Truong hop da do: MLP tai cut 2019, chi bat
A1, macro-F1 doi 0,052 con accuracy doi 0,098, tuc gan gap doi.

Neu chi bao cao mot metric thi **chinh metric quyet dinh confound nao trong nhu
la co ton tai**. O day chay day du phep phan tach tren ca hai va dat canh nhau.

Tai lieu dua ra mot **du doan**. So lieu co quyen bac bo no, va neu bac bo thi
dieu do cung phai duoc noi ra.

In [18]:
metric_frames = {name: frames[("as_reported", name)] for name in METRICS_TO_DECOMPOSE}
metric_comparison = compare_metric_decompositions(metric_frames, controls=CONTROL_NAMES)
pivot = metric_comparison.pivot(index="quantity", columns="metric", values="mean_abs")
pivot["ratio"] = pivot[METRICS_TO_DECOMPOSE[1]] / pivot[METRICS_TO_DECOMPOSE[0]].where(
    pivot[METRICS_TO_DECOMPOSE[0]] > 1e-12)
display(pivot.style.format("{:.4f}"))

print("mean |phi| under each metric; ratio > 1 means the first metric masks it")
for quantity, row in pivot.iterrows():
    ratio = row["ratio"]
    flag = "  <-- MASKED by " + METRICS_TO_DECOMPOSE[0] if ratio == ratio and ratio > 1.5 else ""
    print(f"  {quantity:<16} {METRICS_TO_DECOMPOSE[0]}={row[METRICS_TO_DECOMPOSE[0]]:.4f}  "
          f"{METRICS_TO_DECOMPOSE[1]}={row[METRICS_TO_DECOMPOSE[1]]:.4f}  "
          f"ratio={ratio:.2f}{flag}")

metric,accuracy,macro_f1,ratio
quantity,,,
delta_empty,0.5010,0.4557,1.0994
phi_a1_prior,0.0915,0.0295,3.1004
phi_a2_labels,0.1550,0.1279,1.2119
phi_b1_fs,0.0279,0.0239,1.1655
phi_b2_axis,0.1133,0.0908,1.2478
residual,0.1643,0.2372,0.6926


mean |phi| under each metric; ratio > 1 means the first metric masks it
  delta_empty      macro_f1=0.4557  accuracy=0.5010  ratio=1.10
  phi_a1_prior     macro_f1=0.0295  accuracy=0.0915  ratio=3.10  <-- MASKED by macro_f1
  phi_a2_labels    macro_f1=0.1279  accuracy=0.1550  ratio=1.21
  phi_b1_fs        macro_f1=0.0239  accuracy=0.0279  ratio=1.17
  phi_b2_axis      macro_f1=0.0908  accuracy=0.1133  ratio=1.25
  residual         macro_f1=0.2372  accuracy=0.1643  ratio=0.69


In [19]:
# The document's specific prediction: phi_A1 is near zero under macro-F1 and
# materially non-zero under accuracy. Stated as a test so the data can refuse it.
per_model = []
for name in METRICS_TO_DECOMPOSE:
    block = build_model_summary(frames[("as_reported", name)])
    block = block[["cut", "model"] + PHI_COLUMNS + ["residual"]].copy()
    block.insert(0, "metric", name)
    per_model.append(block)
per_model = pd.concat(per_model, ignore_index=True)
display(per_model.style.format({c: "{:+.4f}" for c in PHI_COLUMNS + ["residual"]}))

,metric,cut,model,phi_a1_prior,phi_a2_labels,phi_b1_fs,phi_b2_axis,residual
0,macro_f1,2015,lightgbm,+0.0161,+0.1539,-0.0276,+0.0789,+0.2171
1,macro_f1,2015,logreg,+0.0269,+0.1436,+0.0101,+0.0465,+0.2466
2,macro_f1,2015,mlp,+0.0278,+0.1562,-0.0085,+0.0519,+0.2208
3,macro_f1,2015,random_forest,+0.0406,+0.1626,-0.0058,+0.0564,+0.1942
4,macro_f1,2017,lightgbm,+0.0294,+0.2315,-0.0026,+0.1441,+0.1267
5,macro_f1,2017,logreg,+0.0494,+0.1686,-0.0156,+0.0792,+0.2937
6,macro_f1,2017,mlp,+0.0479,+0.1958,-0.0733,+0.1230,+0.2197
7,macro_f1,2017,random_forest,+0.0532,+0.1808,-0.0045,+0.1083,+0.2066
8,macro_f1,2019,lightgbm,+0.0281,+0.1928,-0.0667,+0.1302,+0.2577
9,macro_f1,2019,logreg,-0.0047,+0.1729,-0.0141,+0.1390,+0.3036


In [20]:
a1_by_metric = per_model.pivot_table(index=["cut", "model"], columns="metric",
                                     values="phi_a1_prior")
a1_by_metric["abs_gain"] = (a1_by_metric[METRICS_TO_DECOMPOSE[1]].abs()
                            - a1_by_metric[METRICS_TO_DECOMPOSE[0]].abs())
display(a1_by_metric.style.format("{:+.4f}"))

near_zero = a1_by_metric[METRICS_TO_DECOMPOSE[0]].abs()
larger = a1_by_metric[METRICS_TO_DECOMPOSE[1]].abs()
n_masked = int(((near_zero < 0.02) & (larger >= 0.02)).sum())
n_bigger = int((larger > near_zero).sum())
print(f"cells where |phi_A1| is below the 0.02 noise floor under "
      f"{METRICS_TO_DECOMPOSE[0]} but at or above it under "
      f"{METRICS_TO_DECOMPOSE[1]}: {n_masked} of {len(a1_by_metric)}")
print(f"cells where |phi_A1| is simply larger under {METRICS_TO_DECOMPOSE[1]}: "
      f"{n_bigger} of {len(a1_by_metric)}")
print()
if n_masked:
    print("FINDING: the design document's prediction holds. macro-F1 neutralises the")
    print("prior-probability control in these cells, so reporting macro-F1 alone would")
    print("have shown A1 as absent. Both metrics must appear in the paper.")
elif n_bigger > len(a1_by_metric) / 2:
    print("PARTIAL: phi_A1 is consistently larger under accuracy but not hidden below")
    print("the noise floor by macro-F1. The direction predicted holds, the strong form")
    print("does not; say so rather than quoting the strong form.")
else:
    print("REFUSED: the data do not support the document's prediction. macro-F1 does")
    print("not mask phi_A1 here. This must be reported as a correction to section 3/A1,")
    print("not quietly dropped.")

cells where |phi_A1| is below the 0.02 noise floor under macro_f1 but at or above it under accuracy: 4 of 20
cells where |phi_A1| is simply larger under accuracy: 18 of 20

FINDING: the design document's prediction holds. macro-F1 neutralises the
prior-probability control in these cells, so reporting macro-F1 alone would
have shown A1 as absent. Both metrics must appear in the paper.


## 10. So hai co so, tra loi phan bien bang so

Neu hai co so cho ket luan dinh tinh giong nhau thi phan bien ve tinh so sanh
duoc bi vo hieu bang so lieu chu khong bang loi. Neu khac nhau thi do la mot
phat hien phai bao cao, khong duoc giau.

In [21]:
left = frames[("as_reported", PRIMARY_METRIC)].set_index(["cut", "model", "seed"])
right = frames[("common", PRIMARY_METRIC)].set_index(["cut", "model", "seed"])
# Compare only the groups the common basis could actually score; the rest are
# reported above as lost, not treated as agreement.
shared_index = left.index.intersection(right.index)
left, right = left.loc[shared_index], right.loc[shared_index]
quantities = PHI_COLUMNS + ["residual"]
print(f"comparing {len(shared_index)} of {len(frames[('as_reported', PRIMARY_METRIC)])} "
      "instances, the ones the common basis could score")
print("cuts represented:", sorted({int(i[0]) for i in shared_index}))

# Spearman needs at least three instances to mean anything; with fewer it is
# reported as NaN rather than as a number that looks like evidence.
n_instances = len(left)
comparison = pd.DataFrame({
    "mean_abs_diff": [(left[q] - right[q]).abs().mean() for q in quantities],
    "max_abs_diff": [(left[q] - right[q]).abs().max() for q in quantities],
    "sign_agreement": [float((np.sign(left[q]) == np.sign(right[q])).mean())
                       for q in quantities],
    "rank_correlation": [float(left[q].corr(right[q], method="spearman"))
                         if n_instances >= 3 else float("nan")
                         for q in quantities],
}, index=quantities)
display(comparison.style.format({"mean_abs_diff": "{:.4f}", "max_abs_diff": "{:.4f}",
                                 "sign_agreement": "{:.0%}", "rank_correlation": "{:.3f}"}))



comparing 44 of 100 instances, the ones the common basis could score
cuts represented: [2015, 2019, 2021]


,mean_abs_diff,max_abs_diff,sign_agreement,rank_correlation
phi_a1_prior,0.0351,0.0708,59%,0.708
phi_a2_labels,0.1454,0.2608,45%,0.278
phi_b1_fs,0.0327,0.1126,75%,0.380
phi_b2_axis,0.1258,0.3164,23%,-0.383
residual,0.1547,0.4138,95%,0.377


In [22]:
# A bare sign test is brittle for an attribution whose magnitude is near zero,
# so report the disagreement relative to the size of the attribution itself.
comparison["mean_abs_phi"] = [float(left[q].abs().mean()) for q in quantities]
comparison["relative_diff"] = comparison["mean_abs_diff"] / comparison["mean_abs_phi"].where(
    comparison["mean_abs_phi"] > 1e-12)
display(comparison.style.format({"mean_abs_diff": "{:.4f}", "max_abs_diff": "{:.4f}",
                                 "sign_agreement": "{:.0%}", "rank_correlation": "{:.3f}",
                                 "mean_abs_phi": "{:.4f}", "relative_diff": "{:.1%}"}))



,mean_abs_diff,max_abs_diff,sign_agreement,rank_correlation,mean_abs_phi,relative_diff
phi_a1_prior,0.0351,0.0708,59%,0.708,0.0320,109.9%
phi_a2_labels,0.1454,0.2608,45%,0.278,0.1432,101.5%
phi_b1_fs,0.0327,0.1126,75%,0.380,0.0235,139.1%
phi_b2_axis,0.1258,0.3164,23%,-0.383,0.0902,139.5%
residual,0.1547,0.4138,95%,0.377,0.2693,57.4%


In [23]:
for quantity, row in comparison.iterrows():
    if not np.isfinite(row["relative_diff"]):
        status = "NEGLIGIBLE"   # the attribution itself is ~0 on both bases
    elif row["sign_agreement"] == 1.0 and row["relative_diff"] < 0.10:
        status = "STABLE    "
    elif row["rank_correlation"] > 0.9:
        status = "ORDERED   "
    elif n_instances < 3:
        status = "TOO FEW   "
    else:
        status = "UNSTABLE  "
    rho = ("  n/a" if not np.isfinite(row["rank_correlation"])
           else f"{row['rank_correlation']:+.3f}")
    relative = ("  n/a" if not np.isfinite(row["relative_diff"])
                else f"{row['relative_diff']:.0%}")
    print(f"{status} {quantity:<16} sign {row['sign_agreement']:.0%}  "
          f"rank rho {rho}  mean |diff| {relative} of |phi|")

if n_instances < 3:
    print()
    print(f"Only {n_instances} game instance(s): too few for a rank correlation.")
    print("Re-read this section once the full lattice has run.")

print()
print("Reading, per design section 8.5: STABLE means the conclusion does not depend")
print("on the scoring basis. ORDERED means the magnitude ranking survives but the")
print("sign of a near-zero attribution does not, which must be stated in the text.")
print("UNSTABLE is a finding to report, not a result to suppress.")

UNSTABLE   phi_a1_prior     sign 59%  rank rho +0.708  mean |diff| 110% of |phi|
UNSTABLE   phi_a2_labels    sign 45%  rank rho +0.278  mean |diff| 102% of |phi|
UNSTABLE   phi_b1_fs        sign 75%  rank rho +0.380  mean |diff| 139% of |phi|
UNSTABLE   phi_b2_axis      sign 23%  rank rho -0.383  mean |diff| 139% of |phi|
UNSTABLE   residual         sign 95%  rank rho +0.377  mean |diff| 57% of |phi|

Reading, per design section 8.5: STABLE means the conclusion does not depend
on the scoring basis. ORDERED means the magnitude ranking survives but the
sign of a near-zero attribution does not, which must be stated in the text.
UNSTABLE is a finding to report, not a result to suppress.


## 11. Xuat LaTeX cho ban thao

Bang chinh va bang phu luc, dang `booktabs`. Chu thich noi ro tong hang la dong
nhat thuc chu khong phai ket qua do.

In [24]:
headers = {"model": "Model", "cut": "Cut", "delta_empty": r"$\Delta$",
           "residual": "$R$", "residual_share": r"$R/\Delta$"}
headers.update({f"phi_{n}": rf"$\phi_{{{CONTROL_LABELS[n]}}}$" for n in CONTROL_NAMES})
main_columns = ["cut", "model", "delta_empty"] + PHI_COLUMNS + ["residual", "residual_share"]

main_tex = to_booktabs(
    summary, columns=main_columns, headers=headers, escape_headers=False,
    caption=("Shapley-Shorrocks decomposition of the random-versus-temporal "
             "macro-F1 gap, per model and cut-point."),
    label="tab:decomposition",
    note=(r"$\Delta = \sum_i \phi_i + R$ holds by construction: $R$ is defined as "
          r"$\Delta(N)$ and the efficiency axiom gives $\sum_i \phi_i = v(N)$. "
          r"Attributions are signed and are never clipped at zero."),
)
print(main_tex)

\begin{table}[t]
\centering
\caption{Shapley-Shorrocks decomposition of the random-versus-temporal macro-F1 gap, per model and cut-point.}
\label{tab:decomposition}
\begin{tabular}{rlrrrrrrr}
\toprule
Cut & Model & $\Delta$ & $\phi_{A1}$ & $\phi_{A2}$ & $\phi_{B1}$ & $\phi_{B2}$ & $R$ & $R/\Delta$ \\
\midrule
2015 & lightgbm & 0.439 & 0.016 & 0.154 & -0.028 & 0.079 & 0.217 & 0.497 \\
2015 & logreg & 0.474 & 0.027 & 0.144 & 0.010 & 0.047 & 0.247 & 0.520 \\
2015 & mlp & 0.448 & 0.028 & 0.156 & -0.009 & 0.052 & 0.221 & 0.496 \\
2015 & random\_forest & 0.448 & 0.041 & 0.163 & -0.006 & 0.056 & 0.194 & 0.438 \\
2017 & lightgbm & 0.529 & 0.029 & 0.231 & -0.003 & 0.144 & 0.127 & 0.238 \\
2017 & logreg & 0.575 & 0.049 & 0.169 & -0.016 & 0.079 & 0.294 & 0.510 \\
2017 & mlp & 0.513 & 0.048 & 0.196 & -0.073 & 0.123 & 0.220 & 0.436 \\
2017 & random\_forest & 0.544 & 0.053 & 0.181 & -0.005 & 0.108 & 0.207 & 0.379 \\
2019 & lightgbm & 0.542 & 0.028 & 0.193 & -0.067 & 0.130 & 0.258 & 0.476 \\
2019 & l

In [25]:
appendix_tex = to_booktabs(
    lattice_table,
    columns=["config_id", "flags", "active", "m_temporal", "m_reference", "delta", "v"],
    headers={"config_id": "ID", "flags": "A1A2B1B2", "active": "Active controls",
             "m_temporal": "temporal", "m_reference": "reference",
             "delta": r"$\Delta(S)$", "v": "$v(S)$"},
    escape_headers=False,
    caption=f"All sixteen lattice cells, {LATTICE_MODEL} at cut {LATTICE_CUT}.",
    label="tab:lattice",
)
print(appendix_tex)

\begin{table}[t]
\centering
\caption{All sixteen lattice cells, lightgbm at cut 2019.}
\label{tab:lattice}
\begin{tabular}{rllrrrr}
\toprule
ID & A1A2B1B2 & Active controls & temporal & reference & $\Delta(S)$ & $v(S)$ \\
\midrule
0 & 0000 & none & 0.203 & 0.745 & 0.542 & 0.000 \\
1 & 1000 & A1 & 0.249 & 0.745 & 0.496 & 0.046 \\
2 & 0100 & A2 & 0.280 & 0.754 & 0.474 & 0.068 \\
3 & 1100 & A1, A2 & 0.332 & 0.754 & 0.422 & 0.120 \\
4 & 0010 & B1 & 0.131 & 0.745 & 0.614 & -0.072 \\
5 & 1010 & A1, B1 & 0.139 & 0.745 & 0.606 & -0.064 \\
6 & 0110 & A2, B1 & 0.181 & 0.763 & 0.582 & -0.040 \\
7 & 1110 & A1, A2, B1 & 0.181 & 0.763 & 0.582 & -0.040 \\
8 & 0001 & B2 & 0.147 & 0.714 & 0.567 & -0.025 \\
9 & 1001 & A1, B2 & 0.152 & 0.714 & 0.561 & -0.019 \\
10 & 0101 & A2, B2 & 0.406 & 0.637 & 0.231 & 0.311 \\
11 & 1101 & A1, A2, B2 & 0.401 & 0.637 & 0.236 & 0.306 \\
12 & 0011 & B1, B2 & 0.125 & 0.737 & 0.612 & -0.070 \\
13 & 1011 & A1, B1, B2 & 0.144 & 0.737 & 0.593 & -0.050 \\
14 & 0111 & A2, B1, B

## 12. Luu ket qua cho notebook 04

Notebook 04 chan doan phan du nen can dung bang `frame` nay. Ghi vao `results/`
duoi thu muc rieng `shapley_e3` de khong dung file cua E2.

In [26]:
# Never write beside the run's own three output files. The production runner
# owns metrics.parquet, predictions.parquet and run_manifest.json; clobbering
# them costs a two-hour re-run, so this notebook writes only into its own
# subdirectory and refuses to emit any of those names.
PROTECTED = {"metrics.parquet", "predictions.parquet", "run_manifest.json"}
out = RESULTS_DIR / "shapley_e3"
assert out.name not in PROTECTED and out != RESULTS_DIR
out.mkdir(parents=True, exist_ok=True)
for (basis, metric_name), current in frames.items():
    current.to_parquet(out / f"decomposition_{basis}_{metric_name}.parquet", index=False)
    values_by_key[(basis, metric_name)].drop(columns="coalition").to_parquet(
        out / f"coalition_values_{basis}_{metric_name}.parquet", index=False)
summary.to_parquet(out / "model_summary.parquet", index=False)
agreement.to_parquet(out / "agreement_report.parquet", index=False)
written = sorted(f.name for f in out.glob("*.parquet"))
assert not (set(written) & PROTECTED), f"refused: would clobber {set(written) & PROTECTED}"
print(f"wrote {len(written)} files to {out}")
print("run outputs untouched:",
      sorted(f.name for f in RESULTS_DIR.glob("*.parquet")))

wrote 14 files to C:\Users\User\Documents\DAIHOC\IT0001\HK5\In4_Sec\results\shapley_e3
run outputs untouched: ['metrics.parquet', 'predictions.parquet']
